<a href="https://colab.research.google.com/github/PereyraHebe/ISPC-Procesamiento-Habla-2026/blob/main/Laboratorio1_TPH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📥 Instalación de librerías (ejecutar solo una vez)
!pip install ipywidgets librosa numpy matplotlib scipy
# 📚 Importación de módulos
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import ipywidgets as widgets
from ipywidgets import interact
# ⚙️ Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Entorno listo. Puede continuar con la Parte 1.")

Parte 1: La Señal "Analógica" (Referencia)

In [ ]:
# 📝 Parámetros de la señal
frecuencia_original = 50 # Hz (frecuencia de la onda)
duracion = 0.1 # segundos
fs_analogico = 10000 # Hz (muy alto para simular continuidad)
# 📐 Generar eje de tiempo
t = np.linspace(0, duracion, int(fs_analogico * duracion), endpoint=False)
# 🌊 Generar onda senoidal pura (señal analógica simulada)
senal_analogica = np.sin(2 * np.pi * frecuencia_original * t)
# 📊 Visualización
plt.figure(figsize=(10, 4))
plt.plot(t * 1000, senal_analogica, label='Señal "Analógica" (simulada)', linewidth=2)
plt.title('Señal Senoidal de Referencia (50 Hz)')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Amplitud')
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()
# 🔊 Escuchar (opcional, requiere audio habilitado en el navegador)
print("🔊 Reproduciendo señal de referencia...")
display(Audio(senal_analogica, rate=fs_analogico))

Parte 2: Muestreo Interactivo (Sampling)

In [ ]:
def visualizar_muestreo(fs_muestreo):
    # Generar tiempos de muestreo
    t_muestreo = np.arange(0, duracion, 1/fs_muestreo)
    # Tomar muestras de la señal analógica
    muestras = np.sin(2 * np.pi * frecuencia_original * t_muestreo)

    # Gráfico comparativo
    plt.figure(figsize=(10, 4))
    plt.plot(t * 1000, senal_analogica, 'gray', alpha=0.5, label='Señal original')
    plt.stem(t_muestreo * 1000, muestras, linefmt='b-', markerfmt='bo', basefmt='k-',
             label='Muestras tomadas')
    plt.title(f'Muestreo a {fs_muestreo} Hz')
    plt.xlabel('Tiempo (ms)')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # Explicación pedagógica
    if fs_muestreo < 2 * frecuencia_original:
        print("⚠️ ATENCIÓN: fs < 2×f. Estás por debajo del límite de Nyquist. ¡Aparecerá aliasing!")
    else:
        print("✅ Buen muestreo. La frecuencia de muestreo cumple Nyquist (fs ≥ 2×f).")

# 🎛️ Control interactivo
print("🎛️ Use el deslizador para cambiar la frecuencia de muestreo (50 Hz a 200 Hz)")
interact(visualizar_muestreo, fs_muestreo=widgets.IntSlider(min=50, max=200, step=10,
                                                             value=100, description='fs (Hz):'));

Obersevaciones:
1) EN LA MUESTRA DE 100HZ, VEO LOS PUNTOS PEGADOS A LA LINEA DE MUESTRA, Y EENTIENDO QUE HAY UN EQUILIBRIO ENTRE LA SERIE ORIGINAL Y LA MUESTRA, Y QUE CUMPLE LA REGLA DE ORO

2)AL BAJAR LA FECUENCIA A 60Hz (CASI LA MITAD) VEO QUE LOS PUNTOS DE MUESTRA, CAMBIAN, YA NO SE VEN TAN EQUILIBRADOS Y DIFIEREN BASTANTE DE LA ORIGINAL, Y QUE PUEDE ESTAR PERDIENDO INFORMACION EN EN MUESTREO, PRODUCIENDO ALIASING.

3) EN EL 3ER CASO CON 200Hz VEO MAS PUNTOS DE REFERENCIAS, CON RESPECTO A LA MUESTRA ORIGINAL, COMO SI PUEDIERA MUESTRAR DE FORMA MAS SEGURA LA ORIGINAL Y TENER MAYOR FIDELIDAD. CLARO, AL DUPLICAR LA FRECUENCIA DEL PRIMER CASO, SE SIGUE CUMPLIENDO LA REGLA DE ORO.

LO QUE ME LLEVA A PENSAR HAY ALGUN COSTO DE PROCESAMIENTO CON LA 3RA OPCION? PORQUE PARECE MUCHO MEJOR TOMAR ESE MUESTREO, PERO PODRIA COMPLICAR EL ANALISIS EN OTRO SENIDO, O TENER MAYOR COSTO?

4. Parte 3: Cuantización Interactiva (Resolución en Bits)

In [ ]:
def cuantizar_senal(bits):
    # Muestreamos primero a una tasa segura
    fs_seguro = 1000
    t_seguro = np.arange(0, duracion, 1/fs_seguro)
    senal_muestreada = np.sin(2 * np.pi * frecuencia_original * t_seguro)
    # Fórmula de cuantización simétrica (normalizada entre -1 y 1)
    niveles = 2**bits
    señal_cuantizada = np.round(senal_muestreada * (niveles/2 - 1)) / (niveles/2 - 1)
    # Gráfico
    plt.figure(figsize=(10, 4))
    plt.plot(t_seguro * 1000, senal_muestreada, 'gray', alpha=0.6, label='Señal muestreada (continua)')
    plt.plot(t_seguro * 1000, señal_cuantizada, 'r-', linewidth=2, label=f'Cuantizada ({bits} bits)')
    plt.title(f'Cuantización: {bits} bits → {niveles} posibles')
    plt.xlabel('Tiempo (ms)')
    plt.ylabel('Amplitud cuantizada')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    # Explicación
    print(f"📊 Con {bits} bits tenemos {niveles} niveles. "
    f"El 'escalado' visible es el ruido de cuantización.")
# 🎛️ Control interactivo
print("🎛️ Ajuste la profundidad de bits para ver el efecto en la precisión de amplitud")
interact(cuantizar_senal, bits=widgets.IntSlider(min=2, max=8, step=1, value=4,
description='Bits:'));

Parte 4: Aliasing (Solapamiento Frecuencial)

In [ ]:
def demostrar_aliasing():
    """Compara señal original, muestreo insuficiente y reconstrucción errónea"""
    # Frecuencias
    f_alta = 70 # Hz (señal real)
    fs_baja = 80# Hz (frecuencia de muestreo)
    # Tiempos
    t = np.linspace(0, 0.2, 2000, endpoint=False)
    t_muestreo = np.arange(0, 0.2, 1/fs_baja)
    # Señales
    senal_real = np.sin(2 * np.pi * f_alta * t)
    muestras = np.sin(2 * np.pi * f_alta * t_muestreo)
    # Frecuencia que "percibimos" al reconstruir mal (alias)
    f_alias = abs(f_alta - fs_baja) # 70 - 80 = 10 Hz (en valor absoluto)
    senal_alias = np.sin(2 * np.pi * f_alias * t)
    # Gráfico
    plt.figure(figsize=(10, 5))
    plt.plot(t, senal_real, 'b', linewidth=2, label=f'Señal real ({f_alta} Hz)')
    plt.stem(t_muestreo, muestras, linefmt='k-', markerfmt='ko', basefmt=' ',
             label='Muestras')
    plt.plot(t, senal_alias, 'r--', linewidth=2, label=f'Señal reconstruida errónea ({f_alias} Hz)')
    plt.title('⚠️ Efecto Aliasing: Muestreo insuficiente crea frecuencias falsas')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.show()
    print(f"📌 Teorema de Nyquist: fs debe ser ≥ {2*f_alta} Hz para {f_alta} Hz.")
    print(f"📌 Usamos fs = {fs_baja} Hz → ¡INFERIOR AL LÍMITE!")
    print(f"📌 Resultado: El cerebro/computadora interpreta una onda de {f_alias} Hz.")
demostrar_aliasing()

In [ ]:
def demostrar_aliasing():
    """Compara señal original, muestreo insuficiente y reconstrucción errónea"""
    # Frecuencias
    f_alta = 70 # Hz (señal real)
    fs_baja = 140 # Hz (frecuencia de muestreo)
    # Tiempos
    t = np.linspace(0, 0.2, 2000, endpoint=False)
    t_muestreo = np.arange(0, 0.2, 1/fs_baja)
    # Señales
    senal_real = np.sin(2 * np.pi * f_alta * t)
    muestras = np.sin(2 * np.pi * f_alta * t_muestreo)
    # Frecuencia que "percibimos" al reconstruir mal (alias)
    # La fórmula de alias es abs(f_alta - n * fs_baja) donde n es un entero que minimiza el resultado
    # En este caso, si fs_baja = 2 * f_alta, f_alias = f_alta.
    # Si fs_baja es menor, el alias es abs(f_alta - fs_baja) o fs_baja - (f_alta % fs_baja) si f_alta > fs_baja
    # Para este ejemplo, con fs_baja = 140 Hz, f_alias = 70 Hz, lo cual no es aliasing, sino la frecuencia original
    f_alias = abs(f_alta - fs_baja) if fs_baja < 2 * f_alta else f_alta # Modificado para reflejar correctamente el alias o la frecuencia original
    senal_alias = np.sin(2 * np.pi * f_alias * t)
    # Gráfico
    plt.figure(figsize=(10, 5))
    plt.plot(t, senal_real, 'b', linewidth=2, label=f'Señal real ({f_alta} Hz)')
    plt.stem(t_muestreo, muestras, linefmt='k-', markerfmt='ko', basefmt=' ',
             label='Muestras')
    plt.plot(t, senal_alias, 'r--', linewidth=2, label=f'Señal reconstruida errónea ({f_alias} Hz)')
    plt.title('⚠️ Efecto Aliasing: Muestreo insuficiente crea frecuencias falsas')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.show()
    print(f"📌 Teorema de Nyquist: fs debe ser ≥ {2*f_alta} Hz para {f_alta} Hz.")
    if fs_baja < 2 * f_alta:
        print(f"📌 Usamos fs = {fs_baja} Hz → ¡INFERIOR AL LÍMITE!")
        print(f"📌 Resultado: El cerebro/computadora interpreta una onda de {f_alias} Hz.")
    else:
        print(f"📌 Usamos fs = {fs_baja} Hz → ¡IGUAL O SUPERIOR AL LÍMITE DE NYQUIST!")
        print(f"📌 Resultado: El cerebro/computadora interpreta una onda de {f_alias} Hz (correctamente reconstruida).")
demostrar_aliasing()

6. Parte 5: Dominio del Tiempo vs. Dominio de la Frecuencia

In [ ]:
def analisis_fourier_simple():
    """Genera una señal compuesta y muestra su espectro de frecuencias"""
    fs = 1000
    t = np.linspace(0, 1, fs, endpoint=False)
    # Señal compuesta: 50 Hz + 150 Hz + ruido leve
    senal = 0.8 * np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 150 * t) + 0.1 * np.random.randn(len(t))
    # 🔄 Transformada Rápida de Fourier (FFT)
    espectro = np.fft.fft(senal)
    frecuencias = np.fft.fftfreq(len(senal), d=1/fs)
    # Tomamos solo la mitad positiva (el espectro es simétrico)
    idx_pos = frecuencias >= 0
    f_pos = frecuencias[idx_pos]
    amp_pos = np.abs(espectro[idx_pos]) * 2 / len(senal) # Normalización
    # Gráficos lado a lado
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    # Dominio del tiempo
    ax1.plot(t, senal, linewidth=1.5)
    ax1.set_title('📈 Dominio del Tiempo')
    ax1.set_xlabel('Tiempo (s)')
    ax1.set_ylabel('Amplitud')
    ax1.grid(True, alpha=0.3)
    # Dominio de la frecuencia
    ax2.plot(f_pos, amp_pos, color='purple', linewidth=2)
    ax2.set_title('📊 Dominio de la Frecuencia (FFT)')
    ax2.set_xlabel('Frecuencia (Hz)')
    ax2.set_ylabel('Magnitud')
    ax2.set_xlim(0, 200)
    ax2.grid(True, alpha=0.3)
    ax2.axvline(50, color='g', linestyle='--', alpha=0.5, label='50 Hz real')
    ax2.axvline(150, color='r', linestyle='--', alpha=0.5, label='150 Hz real')
    ax2.legend()
    plt.tight_layout()
    plt.show()
    print("🔍 Observe los picos en 50 Hz y 150 Hz. La FFT descompone la señal en sus\n'notas' individuales.")
analisis_fourier_simple()

Actividades para el Estudiante
Responda las siguientes preguntas en un documento aparte o en el mismo notebook.
Justifique sus respuestas con ejemplos del laboratorio.
1. Muestreo: Si graba un tono puro de 300 Hz, ¿cuál es la frecuencia de muestreo
mínima teórica para evitar aliasing? ¿Por qué en la práctica se usa un valor
mayor?
2. Cuantización: Un audio de estudio se graba a 24 bits y 48 kHz. Otro archivo
comprimido para web usa 8 bits y 22 kHz. Describa dos diferencias auditivas y
visuales que esperaría encontrar entre ambos.
3. Aliasing: En el código de la Parte 4, cambie f_alta = 90 y fs_baja = 80. Ejecute
mentalmente o en código: ¿qué frecuencia "falsa" aparecerá? Explique el cálculo.
4. Dominios: Si aplica la FFT a un archivo .wav de voz humana y observa un pico
dominante en 220 Hz, ¿qué información le da esto sobre la señal original? ¿Podría
distinguirse de un silbido de 220 Hz solo con este pico?

1- MUESTREO:
La frecuencia mínima teórica para evitar el aliasing es de 600hz, y en la práctica se usa un mayor valor para tener mayor fidelidad. A continuación, reutilice el código proporcionado y lo edité con los nuevos datos propuestos, para demostrar lo postulado.

In [ ]:
# 📝 Parámetros de la señal
frecuencia_original = 300 # Hz (frecuencia de la onda)
duracion = 0.1 # segundos
fs_analogico = 10000 # Hz (muy alto para simular continuidad)
# 📐 Generar eje de tiempo
t = np.linspace(0, duracion, int(fs_analogico * duracion), endpoint=False)
# 🌊 Generar onda senoidal pura (señal analógica simulada)
senal_analogica = np.sin(2 * np.pi * frecuencia_original * t)
# 📊 Visualización
plt.figure(figsize=(10, 4))
plt.plot(t * 1000, senal_analogica, label='Señal "Analógica" (simulada)', linewidth=2)
plt.title('Señal Senoidal de Referencia (300 Hz)')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Amplitud')
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()
# 🔊 Escuchar (opcional, requiere audio habilitado en el navegador)
print("🔊 Reproduciendo señal de referencia...")
display(Audio(senal_analogica, rate=fs_analogico))

def visualizar_muestreo(fs_muestreo):
    # Generar tiempos de muestreo
    t_muestreo = np.arange(0, duracion, 1/fs_muestreo)
    # Tomar muestras de la señal analógica
    muestras = np.sin(2 * np.pi * frecuencia_original * t_muestreo)

    # Gráfico comparativo
    plt.figure(figsize=(10, 4))
    plt.plot(t * 1000, senal_analogica, 'gray', alpha=0.5, label='Señal original')
    plt.stem(t_muestreo * 1000, muestras, linefmt='b-', markerfmt='bo', basefmt='k-',
             label='Muestras tomadas')
    plt.title(f'Muestreo a {fs_muestreo} Hz')
    plt.xlabel('Tiempo (ms)')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # Explicación pedagógica
    if fs_muestreo < 2 * frecuencia_original:
        print("⚠️ ATENCIÓN: fs < 2×f. Estás por debajo del límite de Nyquist. ¡Aparecerá aliasing!")
    else:
        print("✅ Buen muestreo. La frecuencia de muestreo cumple Nyquist (fs ≥ 2×f).")

# 🎛️ Control interactivo
print("🎛️ Use el deslizador para cambiar la frecuencia de muestreo (100 Hz a 1000 Hz)")
interact(visualizar_muestreo, fs_muestreo=widgets.IntSlider(min=100, max=600, step=10,
                                                             value=600, description='fs (Hz):'));